# Paper B batch 2 — epoch-matched control (6.15) + byte-BPE on ABSA (6.14)
Runtime → **A100** (T4 works, slower). Run top to bottom. Every launch skips runs whose `metrics.json` already exists on Drive, so a disconnect is safe — just rerun. **After any code change set `FORCE=1` for the affected block.**

| Block | Question | Runs | Drive root |
|---|---|---|---|
| F (6.15) | Does the English QA−BIO result survive when BIO trains 7 epochs like joint (not 6)? | XLM-R + RoBERTa × 3 seeds, BIO only (joint copied from `en_bbpe`, same config) = 6 | `IdiomatorRigor_NER_en_epmatch` |
| G (6.14) | Prediction test: byte-BPE stays flat on the most exposed task (ABSA, ~33% punct-follow) while SentencePiece does not | RoBERTa + ModernBERT × 3 seeds × {joint, bio} × 2 domains = 24, then `ow1` BIO 12 | `IdiomatorRigor_NER_absa_{restaurants,laptops}[_ow1]` |

**Predictions (stated before running):** F — XLM-R original gap stays ≈ +0.116 and strip ≈ 0; RoBERTa flat. G — RoBERTa and ModernBERT gap ≈ 0 in every mode in both domains, unlike XLM-R/RemBERT (+0.21 to +0.27 original). If a byte-BPE encoder shows a large gap, "byte-BPE is immune" is dead and that is the finding.

In [ ]:
# 1. Setup: Drive + repo + deps + GPU gate
from google.colab import drive
drive.mount('/content/drive')
%cd /content
!rm -rf /content/repo && git clone https://github.com/JustLetMeBeHello/Idiomator_Research.git /content/repo
%cd /content/repo
!git log --oneline -1
!pip install -q -r Requirements.txt datasets
import torch
assert torch.cuda.is_available(), 'CPU-only runtime (happens after restarts) — Runtime > Change runtime type > GPU'
print('Device: cuda —', torch.cuda.get_device_name(0))

In [ ]:
# 2. Preflight: patches pushed, Drive roots durable + writable. T0 is used later to prove runs are fresh.
import os, json, glob, time, shutil, subprocess
T0 = time.time()
assert 'EPOCHS_BIO' in open('experiments/rigor/run_07_mdeberta_sp_replication.sh').read(), 'old commit — push the run_07 epoch override first'
assert '"modernbert"' in open('experiments/rigor/run_08_extended_gold.py').read(), 'old commit — push run_08 first'
M = '/content/drive/MyDrive'
ROOT_B = f'{M}/IdiomatorRigor_NER_en_bbpe'        # existing English runs (source of the copied joint dirs)
ROOT_E = f'{M}/IdiomatorRigor_NER_en_epmatch'
ABSA = {d: f'{M}/IdiomatorRigor_NER_absa_{d}' for d in ('restaurants','laptops')}
ABSA_OW1 = {d: f'{M}/IdiomatorRigor_NER_absa_{d}_ow1' for d in ('restaurants','laptops')}
for r in [ROOT_B, ROOT_E, *ABSA.values(), *ABSA_OW1.values()]:
    os.makedirs(r, exist_ok=True)
    assert os.path.realpath(r).startswith('/content/drive'), f'{r} is not on Drive'
    open(f'{r}/.probe','w').write('ok'); assert open(f'{r}/.probe').read()=='ok'; os.remove(f'{r}/.probe')
assert os.path.exists('data/ner_wikiann_multi/Splits/test.jsonl')
def readback(root, pat='*/*_s*/test_predictions.jsonl', fresh_only=False):
    for f in sorted(glob.glob(f'{root}/{pat}')):
        mt = os.path.getmtime(f)
        if fresh_only and mt < T0: continue
        print(f'{sum(1 for _ in open(f)):5d} rows  {time.ctime(mt)}  {f[len(root)+1:]}')
print('preflight ok')

## Block F (6.15) — epoch-matched English control
Joint was trained 7 epochs, BIO 6. Retrain BIO at 7 for XLM-R and RoBERTa, 3 seeds; reuse the existing joint runs (identical config, seeds, data) by copying them into a fresh root so nothing existing is overwritten.

In [ ]:
# 3. Copy existing joint runs into the new root (XLM-R under flip/, RoBERTa under bbpe/); verify config + rows
for enc, sub in (('xlmr','flip'), ('roberta','bbpe')):
    for s in (42, 123, 7):
        src, dst = f'{ROOT_B}/{sub}/{enc}_joint_s{s}', f'{ROOT_E}/{sub}/{enc}_joint_s{s}'
        assert os.path.getsize(f'{src}/test_predictions.jsonl') > 0 and os.path.exists(f'{src}/metrics.json'), f'missing source {src}'
        if not os.path.exists(f'{dst}/metrics.json'):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        cfg = json.load(open(f'{src}/config.json')) if os.path.exists(f'{src}/config.json') else {}
        print('✓', f'{enc}_joint_s{s}', {k: cfg.get(k) for k in ('epochs','lr','batch_size') if k in cfg} or '(no config.json persisted)')

In [ ]:
# 4. Launch BIO at 7 epochs (joint skips via metrics.json). 6 runs. Resumable.
FORCE = 0
ENV_EN = "LANGS=English TEST_LANGS=English O_WEIGHT=0.615 LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 EPOCHS_JOINT=7 EPOCHS_BIO=7"
for model, short, sub in [('xlm-roberta-base','xlmr','flip'), ('roberta-base','roberta','bbpe')]:
    !DRIVE_OUT={ROOT_E} DATA_DIR=data/ner_wikiann_multi/Splits {ENV_EN} FORCE={FORCE} SEEDS="42 123 7" OUT_SUB={sub} MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 5. Read back from Drive: BIO dirs must be post-T0 (fresh), 500 rows each; then score. Compare with the 6-epoch run_08 in ROOT_B.
print('--- fresh BIO runs ---'); readback(ROOT_E, '*/*_bio_s*/test_predictions.jsonl', fresh_only=True)
n_fresh = sum(1 for f in glob.glob(f'{ROOT_E}/*/*_bio_s*/test_predictions.jsonl') if os.path.getmtime(f) >= T0)
print('fresh BIO files this session:', n_fresh, '(expect 6 on a first run; 0 means everything was skipped from an earlier session — check mtimes above)')
!BBPE=1 python experiments/rigor/run_08_extended_gold.py --preds-root {ROOT_E} --seeds 42 123 7 --out {ROOT_E}/run_08_ner_en_epmatch_3seed.json --force

## Block G (6.14) — byte-BPE encoders on ABSA-14
RoBERTa + ModernBERT, same data/seeds/LR as the SentencePiece/WordPiece ABSA runs. Original BIO O-weight first (from each domain's `meta.json`), then `ow1` (O_WEIGHT=1.0) BIO only; joint is weight-independent and is copied into the `ow1` root.

In [ ]:
# 6. Build ABSA data + precondition check (same as Batch cell 11)
for dom in ('restaurants','laptops'):
    out = f'data/absa14_{dom}/Splits'
    r = subprocess.run(['python','experiments/rigor/run_33_absa_data_prep.py','--config',dom,'--out-dir',out,'--force'],capture_output=True,text=True)
    print(f'##### {dom}'); print(r.stdout[-600:], r.stderr[-600:])
    meta = json.load(open(f'{out}/meta.json'))
    assert meta['splits']['test']['punct_follow_pct'] > 20, 'precondition missing — wrong (pre-tokenized?) data'
    print('n train/dev/test:', [meta['splits'][s]['n'] for s in ('train','dev','test')], 'o_weight', meta['o_weight'])

In [ ]:
# 7. Dry-run on ABSA (1 epoch, both trainers, both encoders) — fails loudly on a compat problem before 36 runs
dom = 'laptops'; ow = json.load(open(f'data/absa14_{dom}/Splits/meta.json'))['o_weight']
for model, short, trc in [('roberta-base','roberta',0), ('answerdotai/ModernBERT-base','modernbert',1)]:
    !DRIVE_OUT={ABSA[dom]} DATA_DIR=data/absa14_{dom}/Splits LANGS=English TEST_LANGS=English O_WEIGHT={ow} LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 TRUST_REMOTE_CODE={trc} OUT_SUB=bbpe MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh --dry-run
!rm -rf {ABSA[dom]}/bbpe/_dryrun_joint {ABSA[dom]}/bbpe/_dryrun_bio

In [ ]:
# 8. Launch original-weight ABSA: RoBERTa + ModernBERT × seeds 42 123 7 × {joint, bio} × 2 domains (24 runs). Resumable.
FORCE = 0
for dom in ('restaurants','laptops'):
    ow = json.load(open(f'data/absa14_{dom}/Splits/meta.json'))['o_weight']
    for model, short, trc in [('roberta-base','roberta',0), ('answerdotai/ModernBERT-base','modernbert',1)]:
        !DRIVE_OUT={ABSA[dom]} DATA_DIR=data/absa14_{dom}/Splits LANGS=English TEST_LANGS=English O_WEIGHT={ow} LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 TRUST_REMOTE_CODE={trc} FORCE={FORCE} SEEDS="42 123 7" OUT_SUB=bbpe MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 9. ow1: copy the joint runs just produced into the ow1 roots, then train BIO at O_WEIGHT=1.0 (12 runs). Resumable.
FORCE = 0
for dom in ('restaurants','laptops'):
    for enc in ('roberta','modernbert'):
        for s in (42,123,7):
            src, dst = f'{ABSA[dom]}/bbpe/{enc}_joint_s{s}', f'{ABSA_OW1[dom]}/bbpe/{enc}_joint_s{s}'
            assert os.path.exists(f'{src}/metrics.json'), f'missing joint source {src} — cell 8 incomplete'
            if not os.path.exists(f'{dst}/metrics.json'): shutil.copytree(src, dst, dirs_exist_ok=True)
    for model, short, trc in [('roberta-base','roberta',0), ('answerdotai/ModernBERT-base','modernbert',1)]:
        !DRIVE_OUT={ABSA_OW1[dom]} DATA_DIR=data/absa14_{dom}/Splits LANGS=English TEST_LANGS=English O_WEIGHT=1.0 LR_JOINT=3e-5 LR_BIO=3e-5 WORK_LOCAL=1 TRUST_REMOTE_CODE={trc} FORCE={FORCE} SEEDS="42 123 7" OUT_SUB=bbpe MODEL={model} ENC_SHORT={short} bash experiments/rigor/run_07_mdeberta_sp_replication.sh

In [ ]:
# 10. Read back (BIO rows must equal joint rows; fresh = post-T0) then score every root with NEW output names (existing run_08 JSONs are not overwritten)
for tag, roots in (('orig', ABSA), ('ow1', ABSA_OW1)):
    for dom, root in roots.items():
        print(f'\n##### {dom} [{tag}]'); readback(root, 'bbpe/*_s*/test_predictions.jsonl')
        for enc in ('roberta','modernbert'):
            for s in (42,123,7):
                nj = sum(1 for _ in open(f'{root}/bbpe/{enc}_joint_s{s}/test_predictions.jsonl')); nb = sum(1 for _ in open(f'{root}/bbpe/{enc}_bio_s{s}/test_predictions.jsonl'))
                assert nj == nb, f'{dom} {tag} {enc} s{s}: joint {nj} rows vs bio {nb} (BIO skipped examples?)'
        r = subprocess.run(['python','experiments/rigor/run_08_extended_gold.py','--preds-root',root,'--seeds','42','123','7','--out',f'{root}/run_08_absa_{dom}_{tag}_bbpe_3seed.json','--force'],capture_output=True,text=True,env={**os.environ,'BBPE':'1'})
        print(r.stdout[-3500:], r.stderr[-1200:])